# **LaTeK: Large-scale Benchmark Dataset Generation Framework for Evaluating Text-to-KG Generation**

## Setting
#### 1) Configure the following settings:
  - working_path
  - openai_key
  - google_key
  - mistral_key
#### 2) Make the pickle folder under the working path

In [ ]:
import os
import pickle
import time
import numpy as np
import pandas as pd
import json
import re

from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

import wikipediaapi
from datasets import load_dataset
from func_timeout import func_set_timeout
from func_timeout import FunctionTimedOut

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from kg_gen import KGGen

class MyGraph:
    # wikipedia entity name, entity ID, sub ID, triples (List Type)
    def __init__(self, ename, eid, sid, triples):
        self.entity_name = ename
        self.entity_id = eid
        self.sub_id = sid
        self.triple_list = sorted(triples) 
        self.sentences = ['', '', '', '']  # prompt1,2,3,4

    def get_triple_str(self):
        str_triples = [str(t) for t in self.triple_list]
        return "\n".join(str_triples)

    def set_sentence(self, i, sentence):
        self.sentences[i] = sentence
        
    def get_sentence(self, i):
        return self.sentences[i]

    def get_triple_size(self):
        return len(self.triple_list)

    def get_triple_list(self):
        return self.triple_list  

def download_wikidata(wikipedia, entity_name):
  wpage = wikipedia.page(entity_name)
  if wpage.exists():
    return wpage.text
  else:
    return ""

def get_wikidata(eid):
    ifile = open(working_path+"/wiki_data.txt", "rt", encoding="utf-8" )
    cnt = 0
    max_cnt = eid+1
    wiki_text = ''
    wiki_entity = ''
    while (line:=ifile.readline()) != "":
      if line.startswith("<MY_DELIMITER>"):
        if cnt==max_cnt: 
            return wiki_text[:-1]  # -1: remove new line
            break
        cnt = cnt + 1
    
        wiki_entity = line.replace("<MY_DELIMITER>", "").strip()
        wiki_text = ''
    
      else:
        wiki_text = wiki_text + line
    ifile.close()

    if cnt==max_cnt: return wiki_text[:-1] # -1: remove new line
    else: return None

In [ ]:
working_path = 'Your Working Path'
openai_key = ''
google_key= ''
mistral_key = ''
os.environ['OPENAI_API_KEY'] = openai_key
os.environ['GOOGLE_API_KEY'] = google_key
os.environ['MISTRAL_API_KEY'] = mistral_key

num_entities = 5000
doc_segment_size = 20
doc_overlap_size = 2

## Wikipedia Page Collection

In [ ]:
yago = load_dataset("VLyb/YAGO3-10") # https://huggingface.co/datasets/VLyb/YAGO3-10

cnt = 0
entity_map = {}
for r in yago['train']:
    if entity_map.get(r['head']) == None:
        entity_map[r['head']] = cnt
        cnt = cnt + 1
    if entity_map.get(r['tail']) == None:
        entity_map[r['tail']] = cnt
        cnt = cnt + 1
for r in yago['validation']:
    if entity_map.get(r['head']) == None:
        entity_map[r['head']] = cnt
        cnt = cnt + 1
    if entity_map.get(r['tail']) == None:
        entity_map[r['tail']] = cnt
        cnt = cnt + 1
for r in yago['test']:
    # Warning in "VLyb/YAGO3-10": '\ufeffStan_Collymore' should be corrected.
    if r['head'] == '\ufeffStan_Collymore':
        ename = 'Stan_Collymore'
        if entity_map.get(ename) == None:
            entity_map[ename] = cnt
            cnt = cnt + 1
        if entity_map.get(r['tail']) == None:
            entity_map[r['tail']] = cnt
            cnt = cnt + 1   
    else:    
        if entity_map.get(r['head']) == None:
            entity_map[r['head']] = cnt
            cnt = cnt + 1
        if entity_map.get(r['tail']) == None:
            entity_map[r['tail']] = cnt
            cnt = cnt + 1

id_map = {v:k for k,v in entity_map.items()}

with open(working_path+'/pickle/id_map.pickle', 'wb') as file:
    pickle.dump(id_map, file)

In [ ]:
st = time.time()

wikipedia = wikipediaapi.Wikipedia('LaTeK', 'en')
ofile = open(working_path+'/wiki_data.txt', 'wt', encoding='utf-8')
error_ofile = open(working_path+'/ERROR_LIST.txt', 'wt', encoding='utf-8')

for i in range(num_entities):
    c = id_map[i]
    try:
      wdata = ""
      wdata = download_wikidata(wikipedia, c)
    except Exception as e:
      print("# Exception", e, ":", c, "[", wdata, "]")
      # try one more
      try:
        wdata = ""
        wdata = download_wikidata(wikipedia, c)
      except Exception as e2:
        print("# Exception2", e2, ":", c, "[", wdata, "]")

    if wdata!="": ofile.write('<MY_DELIMITER>'+c+'\n'+wdata+'\n')
    else:
        print("NOT_EXISTS>", c, "[", wdata, "]")
        error_ofile.write(c+'\n')
        ofile.write('<MY_DELIMITER>'+c+'\nNOT_EXISTS\n')
    ofile.flush()
    error_ofile.flush()
    if i%1000==0: print('#', i)
ofile.close()
error_ofile.close()

et = time.time()

print("* Wikipedia Download Time:", et-st)

### - Statistics for Wikipedia Pages

In [ ]:
page_sizes = np.zeros(num_entities)

for i in range(num_entities):
    page_sizes[i] = len(get_wikidata(i))

with open(working_path+'/pickle/page_sizes.pickle', 'wb') as file:
    pickle.dump(page_sizes, file)

In [ ]:
page_sizes2 = page_sizes[page_sizes>len("NOT_EXISTS")]
print("|page_sizes2|=", len(page_sizes2))
plt.hist(page_sizes2)

In [ ]:
page_sizes3 = page_sizes[(page_sizes<=10000) & (page_sizes>len("NOT_EXISTS"))]
plt.hist(page_sizes3)

In [ ]:
# Statistics
print("* Statistics:")
print("|Not Available Pages|=", (page_sizes==len("NOT_EXISTS")).sum())
print("|Pages with size>10000|=", (page_sizes>10000).sum())
print("|page_sizes3|=", len(page_sizes3))

## KG Extraction

In [ ]:
kgen = KGGen(
  model="openai/gpt-4o-mini", 
  temperature=0.0,
  api_key=openai_key
) 

@func_set_timeout(120)
def kg_extraction(doc):
    if (len(doc)>10000) or (doc=='NOT_EXISTS') : return None
    graph = kgen.generate(
      input_data=doc,
      temperature=0.0,
      cluster = True
    )
    return graph

st = time.time()

wgraphs = [] # wiki graphs
timeout_cnt = 0

for i in range(0, num_entities):
    try:
        g = kg_extraction(get_wikidata(i))
    except FunctionTimedOut:
        g = None
        print("# timeout", i)
        timeout_cnt = timeout_cnt + 1
    wgraphs.append(g)
    with open(working_path+'/pickle/g'+str(i)+'.pickle', 'wb') as file:
        pickle.dump(g, file)

    if i%100==0: print('#', i)

with open(working_path+'/pickle/g.pickle', 'wb') as file:
    pickle.dump(wgraphs, file)
   
et = time.time()

print('* KG Extraction Time:', et-st)

In [ ]:
triple_sizes = [len(g.relations) for g in wgraphs if g != None]

plt.hist(triple_sizes, bins=np.arange(0,100,5))

In [ ]:
# Statistics
print("* Statistics:")
print("|Pages with Timeout| =", timeout_cnt)
print("|Generated KGs| =", len(triple_sizes))
print("Mean, Max, Min for KG Size =", np.mean(triple_sizes), np.max(triple_sizes), np.min(triple_sizes))

## Entity/Relation Clustering

In [ ]:
############################## Entity Clustering ###############################
node_set = set()

for i in range(num_entities):
    g = wgraphs[i]
    if g==None: continue
    node_set = node_set.union(set([a for a,b,c in g.relations]))
    node_set = node_set.union(set([c for a,b,c in g.relations]))
    
node_list=list(node_set)

############ Node Embedding
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
node_embedding_vectors = np.array(embeddings.embed_documents(node_list))

with open(working_path+'/pickle/node_embedding_vectors.pickle', 'wb') as file:
    pickle.dump(node_embedding_vectors, file)

############ Node Clustering
dbscan_node = DBSCAN(eps=0.3, min_samples=2)
dbscan_node.fit(node_embedding_vectors)

with open(working_path+'/pickle/dbscan_node.pickle', 'wb') as file:
    pickle.dump(dbscan_node, file)

node_map = {}
for i in range(len(node_list)):
    if dbscan_node.labels_[i]==-1:
        node_map[node_list[i]] = node_list[i]

cnt = 0
for i in [int(cid) for cid in set(dbscan_node.labels_)]:
    if i==-1: continue
    
    elements = np.where(dbscan_node.labels_==i)[0]
    print("* node clusters:", [node_list[e] for e in elements]);
    first_tok = elements[0]

    for j in elements:
        node_map[node_list[j]] = node_list[first_tok]
 
with open(working_path+'/pickle/node_map.pickle', 'wb') as file:
    pickle.dump(node_map, file)
with open(working_path+'/pickle/node_list.pickle', 'wb') as file:
    pickle.dump(node_list, file)

    
############################## Edge Clustering ###############################
edge_set = set()

for i in range(num_entities):
    g = wgraphs[i]
    if g==None: continue
    edge_set = edge_set.union(set([b for a,b,c in g.relations]))
    
edge_list=list(edge_set)

############ Edge Embedding
embeddings2 = OpenAIEmbeddings(model="text-embedding-3-small")
edge_embedding_vectors = np.array(embeddings2.embed_documents(edge_list))

with open(working_path+'/pickle/edge_embedding_vectors.pickle', 'wb') as file:
    pickle.dump(edge_embedding_vectors, file)

############ Edge Clustering
dbscan_edge = DBSCAN(eps=0.3, min_samples=2)
dbscan_edge.fit(edge_embedding_vectors)

with open(working_path+'/pickle/dbscan_edge.pickle', 'wb') as file:
    pickle.dump(dbscan_edge, file)

edge_map = {}
for i in range(len(edge_list)):
    if dbscan_edge.labels_[i]==-1:
        edge_map[edge_list[i]] = edge_list[i]

for i in [int(cid) for cid in set(dbscan_edge.labels_)]:
    if i==-1: continue
    
    elements = np.where(dbscan_edge.labels_==i)[0]
    print("* edge clusters:", [edge_list[e] for e in elements]);
    first_tok = elements[0]

    for j in elements:
        edge_map[edge_list[j]] = edge_list[first_tok]

with open(working_path+'/pickle/edge_map.pickle', 'wb') as file:
    pickle.dump(edge_map, file)
with open(working_path+'/pickle/edge_list.pickle', 'wb') as file:
    pickle.dump(edge_list, file)

In [ ]:
############## Transforming Relations
def trans_rel(nodemap, edgemap, triples):
    new_triples = []
    for (h,r,t) in triples:
        if nodemap.get(h)==None: print("# Error1:", h)
        if nodemap.get(t)==None: print("# Error2:", t)
        if edgemap.get(r)==None: print("# Error3:", r)        
        tr = (nodemap.get(h), edgemap.get(r), nodemap.get(t))
        new_triples.append(tr)
    return new_triples

tr_graphs = []
for g in wgraphs:
    if g==None: tr_graphs.append(None)
    else: 
        tr_graphs.append(trans_rel(node_map, edge_map, list(g.relations)))

## KG Split

In [ ]:
total_num = 0
benchmark_graphs=[]
np.random.seed(0)

def split_docs(relations, segment_size, overlap_size):
    total_size = len(relations)
    
    perm = np.random.permutation(total_size)
    relations = [relations[i] for i in perm]
    
    if total_size<=segment_size:
        return [relations]
    
    docs = []
    for i in range(0, total_size, segment_size-overlap_size):
        if i+segment_size > total_size:
            if i+segment_size-overlap_size < total_size:
                docs.append(relations[i:]+relations[:segment_size - len(relations[i:])])
            else:
                docs.append(relations[i:]+relations[:overlap_size])
        else: 
            docs.append(relations[i:i+segment_size])
    return docs

for eid, g in enumerate(wgraphs):
    if g==None: continue
    tr = list(g.relations) # triples
        
    partitions = split_docs(tr, doc_segment_size, doc_overlap_size)
    total_num = total_num + len(partitions)

    for sub_id, p in enumerate(partitions):
        new_graph = MyGraph(id_map[eid], eid, sub_id, p)  # (ename, eid, sid, triples):
        benchmark_graphs.append(new_graph)

print('* The number of benchmark data:', total_num)

with open(working_path+'/pickle/benchmark_g_without_text.pickle', 'wb') as file:
    pickle.dump(benchmark_graphs, file)

## Text Generation

In [ ]:
#####################################################################
batch_size = 100
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.0)
#####################################################################

prompt1 = '''Create sentences with the following triples whose form is (subject, property, object).

# Requirements #
1) Generate sentences talkatively with simple sentence structures.
2) Do not include line numbers and special characters including a new line.

# Triples #
%s'''

prompt2 = '''Rephrase the following sentences.

# Requirement #
- Do not include line numbers and special characters including a new line.

# Sentences to be rephrased #
%s'''

prompt3 = '''Create sentences with the following triples whose form is (subject, property, object).

# Requirements #
1) Generate sentences talkatively with complex sentence structures.
2) Do not include line numbers and special characters including a new line.

# Triples #
%s'''

    
prompt4 = '''Rephrase the following sentences.

# Requirement #
- Do not include line numbers and special characters including a new line.

# Sentences to be rephrased #
%s'''

prompt_list = [prompt1, prompt2, prompt3, prompt4]

In [ ]:
st = time.time()

for p in range(0,4):
    prompt = prompt_list[p]    
    
    for i in range(0, len(benchmark_graphs), batch_size):
        if i+batch_size<len(benchmark_graphs): end_idx = i+batch_size
        else: end_idx = len(benchmark_graphs)
        
        question_list = []
        for j in range(i, end_idx): # batch
            if p%2==0:
                question_list.append(prompt % (benchmark_graphs[j].get_triple_str()))
            else:
                question_list.append(prompt % (benchmark_graphs[j].get_sentence(p-1)))
            
        results = llm.batch(question_list)
    
        for idx, r in enumerate(results):
            benchmark_graphs[i+idx].set_sentence(p, r.content)            

with open(working_path+'/pickle/benchmark_g.pickle', 'wb') as file:
    pickle.dump(benchmark_graphs, file)

et = time.time()

print("* Text Generation Time from KG:", et-st)

## Metadata Extraction

In [ ]:
# https://yago-knowledge.org/downloads/yago-3
df = pd.read_csv(working_path+'/yagoTypes.tsv', header=None, skiprows=[0], sep='\t')
tydf = df.iloc[:, 1:4].copy()
tydf.rename(columns={1:'h', 2:'r', 3:'t'}, inplace=True)

selected_entity_map1 = {}
for i in range(num_entities):
    if wgraphs[i] != None:
        ename = id_map[i]
        selected_entity_map1['<'+ename+'>'] = i

mytydf = tydf[tydf.apply(lambda r: selected_entity_map1.get(r.h)!=None, axis=1)].copy()

selected_entity_map2 = {ty:idx+10000000 for idx, ty in enumerate(mytydf['t'].unique())}

mytydf['hid'] = mytydf.h.apply(lambda x: selected_entity_map1.get(x))
mytydf['tid'] = mytydf.t.apply(lambda x: selected_entity_map2.get(x))
mytydf['mytype']='TYPE_RELATION'

In [ ]:
df1 = yago['train'].to_pandas()
df2 = yago['validation'].to_pandas()
df3 = yago['test'].to_pandas()
triple_df = pd.concat([df1, df2, df3], ignore_index=True)
triple_df.rename(columns={'head':'h', 'relation':'r', 'tail':'t'}, inplace=True)
my_triple_df =  triple_df[triple_df.apply(lambda r: (selected_entity_map1.get('<'+r.h+'>')!=None) 
                                                and (selected_entity_map1.get('<'+r.t+'>')!=None), axis=1)].copy()

In [ ]:
my_triple_df['hid'] = my_triple_df.h.apply(lambda x: selected_entity_map1.get('<'+x+'>'))
my_triple_df['tid'] = my_triple_df.t.apply(lambda x: selected_entity_map1.get('<'+x+'>'))
my_triple_df['mytype'] = 'RELATION'

In [ ]:
# entity nodes
neo4j_nodedf1 = pd.DataFrame(columns = ['name', 'nodeID:ID'], data = selected_entity_map1.items())
neo4j_nodedf1 [':LABEL'] = 'INSTANCE'

# type nodes
neo4j_nodedf2 = pd.DataFrame(columns = ['name', 'nodeID:ID'], data = selected_entity_map2.items())
neo4j_nodedf2 [':LABEL'] = 'CLASS'

neo4j_nodedf = pd.concat([neo4j_nodedf1, neo4j_nodedf2], ignore_index=True)
neo4j_nodedf.to_csv(working_path+'/nodes.csv', sep=',', index=False)

neo4j_relationdf = pd.concat([mytydf[['hid', 'r', 'tid', 'mytype']],
                              my_triple_df[['hid', 'r', 'tid', 'mytype']]], ignore_index=True)
neo4j_relationdf.rename(columns={'hid':':START_ID', 'r':'name', 'tid':':END_ID', 'mytype':':TYPE'}, inplace=True)
neo4j_relationdf.to_csv(working_path+'/relationships.csv', sep=',', index=False)

### Dataset Saving

In [ ]:
text_llist = [[],[],[],[],[]]
for idx, g in enumerate(benchmark_graphs):
    for i in range(4):
        text_llist[i].append(g.get_sentence(i))
    text_llist[4].append(
        {'gid':idx, 'entity_id':g.entity_id, 'entity_name': g.entity_name, 
         'triple_set':g.get_triple_list()})

json_str = {'type1':text_llist[0], 'type2':text_llist[1], 
            'type3':text_llist[2], 'type4':text_llist[3], 'answers':text_llist[4]}

with open(working_path+'/benchmark_v1.json', 'wt') as file:
    json.dump(json_str, file, indent=4)